In [1]:
import faiss
import numpy as np
import pandas as pd

from tqdm import tqdm

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
chunks_df = pd.read_pickle("../data/chunks.pkl")

embeddings = np.load("../data/embeddings.npy")

index = faiss.read_index("../data/faiss_index.bin")

print(chunks_df.shape)
print(embeddings.shape)

(556, 6)
(556, 384)


In [3]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
evaluation_df = pd.read_csv(
    "../data/evaluation_dataset.csv"
)

evaluation_df.head()

,course,file_name,page,question_type,question,ground_truth,context
0,AI Tools,Lec4 Convolution Neural Networks.pdf,2,Definition,What is a feature vector in the context of CNN?,A set of high-level features extracted by each...,Convolution Neural Networks\nImagine a factory...
1,AI Tools,Lec4 Convolution Neural Networks.pdf,2,Concept,What is the purpose of each layer in a CNN pro...,To extract a different set of high-level features,Convolution Neural Networks\nImagine a factory...
2,AI Tools,Lec4 Convolution Neural Networks.pdf,2,Application,What is the output of a CNN production line?,Class labels,Convolution Neural Networks\nImagine a factory...
3,AI Tools,Lec4 Convolution Neural Networks.pdf,28,Definition,What is FC?,Fully Connected layers,Convolutional Neural Networks CNN\nWhy fully c...
4,AI Tools,Lec4 Convolution Neural Networks.pdf,28,Function,What does Softmax do?,Converts scores to probabilities that sum to 1,Convolutional Neural Networks CNN\nWhy fully c...


In [5]:
def retrieve(query, k=5):

    query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
    ).astype("float32")

    distances, indices = index.search(query_embedding, k)

    results = chunks_df.iloc[indices[0]].copy()

    results["distance"] = distances[0]

    return results

In [6]:
question = evaluation_df.iloc[0]["question"]

retrieve(question)

,chunk_id,course,file_name,page,chunk_text,embedding,distance
104,105,AI Tools,Lec3 CV DIP Fundamentals PI II.pdf,37,Feature Vector vs. Feature Space • Feature vec...,"[-0.045262106, -0.027007705, 0.024864135, -0.0...",0.694118
105,106,AI Tools,Lec3 CV DIP Fundamentals PI II.pdf,38,Feature Vector vs. Feature Space,"[-0.03658646, -0.048241567, 0.06453859, 0.0167...",0.602405
216,217,AI Tools,Lec7 LLM vs. GAN LMS.pdf,24,Convolutional Neural Networks CNN Stacking Lay...,"[-0.05144697, -0.11790486, 0.044429373, 0.0075...",0.552589
141,142,AI Tools,Lec4 Convolution Neural Networks.pdf,27,Convolutional Neural Networks CNN Stacking Lay...,"[-0.05144697, -0.11790486, 0.044429373, 0.0075...",0.552589
103,104,AI Tools,Lec3 CV DIP Fundamentals PI II.pdf,36,Features Feature: A piece of information about...,"[0.06179624, 0.018864272, -0.0017190735, 0.040...",0.543513


In [7]:
def hit_rate_at_k(
    ground_truth_page,
    retrieved_pages
):

    return int(
        ground_truth_page in retrieved_pages
    )

In [8]:
hits = []

for _, row in tqdm(
    evaluation_df.iterrows(),
    total=len(evaluation_df)
):

    retrieved = retrieve(
        row["question"],
        k=5
    )

    retrieved_pages = retrieved[
        "page"
    ].tolist()

    hits.append(

        hit_rate_at_k(

            row["page"],

            retrieved_pages

        )

    )

evaluation_df["hit@5"] = hits

100%|██████████████████████████████████████████████████████████████████████████████████| 60/60 [00:00<00:00, 76.72it/s]


In [9]:
hit_rate = evaluation_df[
    "hit@5"
].mean()

print(
    f"Hit Rate@5 = {hit_rate:.2%}"
)

Hit Rate@5 = 76.67%


In [10]:
def reciprocal_rank(ground_truth_page, retrieved_pages):

    if ground_truth_page in retrieved_pages:

        rank = retrieved_pages.index(ground_truth_page) + 1

        return 1 / rank

    return 0

In [11]:
rr_scores = []

for _, row in tqdm(
    evaluation_df.iterrows(),
    total=len(evaluation_df)
):

    retrieved = retrieve(
        row["question"],
        k=5
    )

    retrieved_pages = retrieved["page"].tolist()

    rr = reciprocal_rank(
        row["page"],
        retrieved_pages
    )

    rr_scores.append(rr)

evaluation_df["RR"] = rr_scores

100%|█████████████████████████████████████████████████████████████████████████████████| 60/60 [00:00<00:00, 108.33it/s]


In [12]:
mrr = evaluation_df["RR"].mean()

print(f"MRR@5 = {mrr:.3f}")

MRR@5 = 0.604
